In [52]:
import re
import time
import random
from datetime import datetime
from typing import Optional, Dict, Any, List

import pandas as pd
import requests

In [53]:
###### CARREGAMENTO TEMPORÁRIO - PROBLEMAS NO SHAREPOINT DO MGI (MIGRAÇÃO) ######
df_onedrive = pd.read_excel('./data/Planilha_PL_CGNOR.xlsx')

In [54]:
#Promover a primeira linha a cabeçalho
df_onedrive.columns = df_onedrive.iloc[0]
df_onedrive = df_onedrive.iloc[1:].reset_index(drop=True)

In [55]:
#Normalizar nomes das colunas
df_onedrive.columns = df_onedrive.columns.str.strip()

In [56]:
#Verificando resultado da normalizacao
for c in df_onedrive.columns:
    print(repr(c))

'Nº'
'Processo'
'Projeto de LEI'
'Descrição'
'Encaminhamento prévio - Teams/E-mail'
'1º Encaminhamento - Rementente'
'1º Encaminhamento - Despacho /Ofício'
'1º Encaminhamento - Data'
'1º Encaminhamento - Prazo p/ Resposta'
'2º Encaminhamento - Remetente'
'2º Encaminhamento - Despacho'
'2º Encaminhamento - Data'
'2º Encaminhamento - Prazo p/ Resposta'
'3º Encaminhamento - Remetente'
'3º Encaminhamento - Despacho'
'3º Encaminhamento - Data'
'3º Encaminhamento - Prazo p/ Resposta'
'Período de elaboração da Análise - Quantidade de dias úteis'
'Envio para a Delog - Data'
'Envio ao Gabinete da Seges - Data'
'Validação pelo gabinete da Seges - Quantidade de dias úteis'
'Assinaturas - Data'
'Manifestação da Seges - Nota técnica'
'Andamento do PL - Data/Detalhamento'
'Observações andamento - Parecer'


In [57]:
df_onedrive = df_onedrive.drop_duplicates().reset_index(drop=True)

In [58]:
# Parser Legislativo (regex)

class LegislativeReferenceParser:
    """
    Extrai e normaliza referências legislativas para o padrão:
    SIGLA NUMERO/ANO

    Exemplos:
    - 'PL nº 4603/2023' -> 'PL 4603/2023'
    - 'Projeto de Lei nº 3.117, de 2024' -> 'PL 3117/2024'
    - 'PDL nº 4/2024' -> 'PDL 4/2024'
    - 'Medida Provisória nº 1.221/2024' -> 'MPV 1221/2024'
    """

    MAPA_SIGLAS = [
        (r"\bPROJETO DE LEI COMPLEMENTAR\b", "PLC"),
        (r"\bPROJETO DE DECRETO LEGISLATIVO\b", "PDL"),
        (r"\bPROJETO DE LEI\b", "PL"),
        (r"\bPLEI\b", "PL"),
        (r"\bMEDIDA PROVISORIA\b", "MPV"),
        (r"\bMEDIDA PROVISÓRIA\b", "MPV"),
        (r"\bPLN\b", "PLN"),
        (r"\bPLC\b", "PLC"),
        (r"\bPDL\b", "PDL"),
        (r"\bPL\b", "PL"),
        (r"\bMPV\b", "MPV"),
        (r"\bMP\b", "MPV"),
    ]

    @classmethod
    def extract_reference(cls, texto: Any) -> Dict[str, Any]:
        if pd.isna(texto):
            return {
                "sigla": pd.NA,
                "numero": pd.NA,
                "ano": pd.NA,
                "proposicao_normalizada": pd.NA,
            }

        texto_limpo = str(texto).strip().upper()

        # normalização básica
        texto_limpo = texto_limpo.replace("Nº", " ")
        texto_limpo = texto_limpo.replace("N°", " ")
        texto_limpo = texto_limpo.replace("NO", " ")
        texto_limpo = texto_limpo.replace("°", " ")
        texto_limpo = texto_limpo.replace("º", " ")
        texto_limpo = re.sub(r"\s+", " ", texto_limpo).strip()

        sigla_encontrada = pd.NA
        for padrao, sigla in cls.MAPA_SIGLAS:
            if re.search(padrao, texto_limpo):
                sigla_encontrada = sigla
                break

        # número + ano
        match = re.search(
            r"(\d{1,3}(?:\.\d{3})*|\d+)\s*(?:/|,\s*DE\s+|,\s*| DE\s+)(\d{4})",
            texto_limpo,
        )

        if not match:
            return {
                "sigla": sigla_encontrada,
                "numero": pd.NA,
                "ano": pd.NA,
                "proposicao_normalizada": pd.NA,
            }

        numero_bruto = match.group(1)
        ano = int(match.group(2))
        numero = int(numero_bruto.replace(".", ""))

        if pd.isna(sigla_encontrada):
            return {
                "sigla": pd.NA,
                "numero": numero,
                "ano": ano,
                "proposicao_normalizada": pd.NA,
            }

        proposicao_normalizada = f"{sigla_encontrada} {numero}/{ano}"

        return {
            "sigla": sigla_encontrada,
            "numero": numero,
            "ano": ano,
            "proposicao_normalizada": proposicao_normalizada,
        }

In [59]:
# Cliente Base HTTP com retry/ backoff

class BaseApiClient:
    def __init__(
        self,
        base_url: str,
        default_timeout: int = 20,
        max_tries: int = 5,
        user_agent: str = "NID-DELOG-Monitor/1.0",
    ):
        self.base_url = base_url.rstrip("/")
        self.default_timeout = default_timeout
        self.max_tries = max_tries
        self.session = requests.Session()
        self.user_agent = user_agent

    def _request_json(
        self,
        url: str,
        params: Optional[dict] = None,
        headers: Optional[dict] = None,
        timeout: Optional[int] = None,
    ) -> Dict[str, Any]:
        params = dict(params or {})
        headers_final = {
            "User-Agent": self.user_agent,
            "Accept": "application/json",
        }
        if headers:
            headers_final.update(headers)

        last_exc = None
        timeout = timeout or self.default_timeout

        for attempt in range(1, self.max_tries + 1):
            try:
                resp = self.session.get(
                    url,
                    params=params,
                    headers=headers_final,
                    timeout=timeout,
                )

                if resp.status_code in (429, 500, 502, 503, 504):
                    raise requests.HTTPError(f"HTTP {resp.status_code}", response=resp)

                resp.raise_for_status()
                return resp.json()

            except (requests.Timeout, requests.ConnectionError, requests.HTTPError, ValueError) as e:
                last_exc = e
                if attempt == self.max_tries:
                    break

                sleep_s = (2 ** (attempt - 1)) * 0.5 + random.uniform(0, 0.4)
                time.sleep(sleep_s)

        raise last_exc

    @staticmethod
    def _fmt_date(iso: Any) -> str:
        if not iso or pd.isna(iso):
            return ""
        s = str(iso).strip().replace("T", " ").replace("Z", "")
        if "." in s:
            s = s.split(".")[0]
        for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
            try:
                return datetime.strptime(s, fmt).strftime("%d/%m/%Y")
            except Exception:
                continue
        return str(iso)

    @staticmethod
    def _deep_get_first(obj: Any, key: str):
        if isinstance(obj, dict):
            if key in obj:
                return obj.get(key)
            for v in obj.values():
                found = BaseApiClient._deep_get_first(v, key)
                if found is not None:
                    return found
        elif isinstance(obj, list):
            for item in obj:
                found = BaseApiClient._deep_get_first(item, key)
                if found is not None:
                    return found
        return None

In [ ]:
# Cliente Camara

class CamaraClient(BaseApiClient):
    def __init__(self):
        super().__init__(
            base_url="https://dadosabertos.camara.leg.br/api/v2",
            default_timeout=15,
            max_tries=5,
        )
        self._deputado_cache = {}

    @staticmethod
    def _extract_deputado_id(uri: str):
        if not isinstance(uri, str):
            return None
        m = re.search(r"/deputados/(\d+)", uri)
        return int(m.group(1)) if m else None

    def _get(self, path: str, params: Optional[dict] = None) -> Dict[str, Any]:
        params = dict(params or {})
        params.setdefault("formato", "json")
        return self._request_json(f"{self.base_url}/{path.lstrip('/')}", params=params)

    def _resolve_partido_uf(self, deputado_id: Optional[int]) -> tuple[str, str]:
        if not deputado_id:
            return "", ""

        if deputado_id in self._deputado_cache:
            dep = self._deputado_cache[deputado_id]
        else:
            dep = self._get(f"deputados/{deputado_id}").get("dados", {}) or {}
            self._deputado_cache[deputado_id] = dep

        ultimo = dep.get("ultimoStatus", {}) or {}
        return (
            (ultimo.get("siglaPartido") or "").strip(),
            (ultimo.get("siglaUf") or "").strip(),
        )

    def fetch(self, sigla: str, numero: Any, ano: Any) -> Dict[str, Any]:
        if pd.isna(sigla) or pd.isna(numero) or pd.isna(ano):
            return {}

        sigla = str(sigla).strip().upper()
        numero = str(int(numero)) if pd.notna(numero) else ""
        ano = str(int(ano)) if pd.notna(ano) else ""

        try:
            busca = self._get(
                "proposicoes",
                params={"siglaTipo": sigla, "numero": numero, "ano": ano},
            )
            dados = busca.get("dados", []) or []
            if not dados:
                return {}

            proposicao = dados[0]
            id_prop = proposicao.get("id")
            if not id_prop:
                return {}

            det = self._get(f"proposicoes/{id_prop}").get("dados", {}) or {}
            status = det.get("statusProposicao", {}) or {}

            autores = self._get(f"proposicoes/{id_prop}/autores").get("dados", []) or []
            autores = sorted(
                autores,
                key=lambda a: (
                    a.get("ordemAssinatura") is None,
                    a.get("ordemAssinatura", 10**9),
                ),
            )

            nomes = []
            partido = ""
            estado = ""
            deputado_id = None

            for a in autores:
                nome = (a.get("nome") or a.get("nomeAutor") or "").strip()
                if nome:
                    nomes.append(nome)

                if not partido:
                    partido = (a.get("siglaPartido") or "").strip()
                if not estado:
                    estado = (a.get("siglaUf") or "").strip()

                if deputado_id is None:
                    deputado_id = self._extract_deputado_id(a.get("uri") or "")

            if deputado_id and (not partido or not estado):
                partido_dep, estado_dep = self._resolve_partido_uf(deputado_id)
                partido = partido or partido_dep
                estado = estado or estado_dep

            if deputado_id is None and nomes:
                partido = partido or "N/A"
                estado = estado or "N/A"

            propositor = ""
            if nomes:
                propositor = "; ".join(nomes[:5])
                if len(nomes) > 5:
                    propositor += f" (+{len(nomes)-5})"

            tramitacoes = self._get(f"proposicoes/{id_prop}/tramitacoes").get("dados", []) or []

            parecer_data = ""
            parecer_orgao = ""
            parecer_despacho = ""
            parecer_link = ""

            candidatos = []
            for t in tramitacoes:
                cod = str(t.get("codTipoTramitacao", ""))
                txt = f"{t.get('descricaoTramitacao','')} {t.get('despacho','')}".lower()
                if cod == "322" or "parecer" in txt:
                    candidatos.append(t)

            if candidatos:
                def parse_dt(item):
                    try:
                        return datetime.fromisoformat(
                            str(item.get("dataHora", "")).replace("Z", "")
                        )
                    except Exception:
                        return datetime.min

                candidatos.sort(key=parse_dt, reverse=True)
                t = candidatos[0]

                parecer_data = self._fmt_date(t.get("dataHora"))
                parecer_orgao = (t.get("siglaOrgao") or "").strip()
                parecer_despacho = (t.get("despacho") or t.get("descricaoTramitacao") or "").strip()
                parecer_link = (t.get("url") or "").strip()

                if not parecer_link:
                    for d in (t.get("documentos") or []):
                        parecer_link = (
                            d.get("url")
                            or d.get("urlInteiroTeor")
                            or d.get("uri")
                            or ""
                        )
                        if parecer_link:
                            break

            link_ficha = f"https://www.camara.leg.br/proposicoesWeb/fichadetramitacao?idProposicao={id_prop}"
            link_inteiro_teor_pl = (det.get("urlInteiroTeor") or "").strip()
            emendas = f"https://www.camara.leg.br/proposicoesWeb/prop_emendas?idProposicao={id_prop}&subst=0"
            substitutivos = f"https://www.camara.leg.br/proposicoesWeb/prop_pareceres_substitutivos_votos?idProposicao={id_prop}"

            return {
                "camara_id_proposicao": str(id_prop),
                "camara_projeto": f"{sigla} {numero}/{ano}",
                "camara_ementa": (proposicao.get("ementa") or "").strip(),
                "camara_data_ultima_tramitacao": self._fmt_date(status.get("dataHora")),
                "camara_orgao_ultima_tramitacao": (status.get("siglaOrgao") or "").strip(),
                "camara_descricao_tramitacao": (status.get("descricaoTramitacao") or "").strip(),
                "camara_situacao_ultima_tramitacao": (status.get("descricaoSituacao") or "").strip(),
                "camara_despacho_ultima_tramitacao": (status.get("despacho") or "").strip(),
                "camara_data_parecer_aprovado": parecer_data,
                "camara_orgao_parecer": parecer_orgao,
                "camara_despacho_parecer": parecer_despacho,
                "camara_link_inteiro_teor_parecer": parecer_link,
                "camara_link_inteiro_teor_pl": link_inteiro_teor_pl,
                "camara_link_ficha_tramitacao": link_ficha,
                "camara_data_proposta_pl": self._fmt_date(det.get("dataApresentacao")),
                "camara_propositor_pl": propositor,
                "camara_partido": partido,
                "camara_estado": estado,
                "camara_emendas": emendas,
                "camara_substitutivos": substitutivos,
            }

        except Exception:
            return {}

In [61]:
#Cliente Senado
class SenadoClient(BaseApiClient):
    def __init__(self):
        super().__init__(
            base_url="https://legis.senado.leg.br/dadosabertos",
            default_timeout=20,
            max_tries=6,
        )
        self._id_cache = {}
        self._detail_cache = {}

    def _get(self, path: str, params: Optional[dict] = None) -> Dict[str, Any]:
        return self._request_json(f"{self.base_url}/{path.lstrip('/')}", params=params or {})

    def get_processo_id(self, sigla: str, numero: Any, ano: Any) -> Optional[str]:
        if pd.isna(sigla) or pd.isna(numero) or pd.isna(ano):
            return None

        sigla = str(sigla).strip().upper()
        numero = str(int(numero)) if pd.notna(numero) else ""
        ano = str(int(ano)) if pd.notna(ano) else ""

        key = (sigla, numero, ano)
        if key in self._id_cache:
            return self._id_cache[key]

        try:
            js = self._get(
                "processo.json",
                params={"sigla": sigla, "numero": numero, "ano": ano, "v": "1"},
            )
            pid = self._deep_get_first(js, "id")
            pid = str(pid).strip() if pid is not None else None
            if pid and pid.endswith(".0"):
                pid = pid[:-2]

            self._id_cache[key] = pid
            return pid
        except Exception:
            self._id_cache[key] = None
            return None


    def _concat_sigla_nome(self, sigla: Any, nome: Any) -> str:
        sigla = str(sigla).strip() if sigla is not None and not pd.isna(sigla) else ""
        nome = str(nome).strip() if nome is not None and not pd.isna(nome) else ""

        if sigla and nome:
            return f"{sigla} - {nome}"
        if sigla:
            return sigla
        if nome:
            return nome
        return ""


    def _extract_last_movement(self, js: Dict[str, Any]) -> tuple[str, str, str]:
        data_ult = ""
        orgao_ult = ""
        situacao_ult = ""

        autuacoes = js.get("autuacoes") or []
        aut0 = autuacoes[0] if isinstance(autuacoes, list) and autuacoes else {}

        informes = aut0.get("informesLegislativos") or []
        if isinstance(informes, list) and informes:
            best = None
            best_dt = None

            for it in informes:
                dt = self._parse_dt_flex(it.get("data"))
                if dt and (best_dt is None or dt > best_dt):
                    best_dt = dt
                    best = it

            if best and best_dt:
                data_ult = best_dt.strftime("%d/%m/%Y")
                situacao_ult = (best.get("descricao") or "").strip()

                # 1) PRIORIDADE: colegiado
                coleg = best.get("colegiado") or {}
                orgao_ult = self._concat_sigla_nome(
                    coleg.get("sigla"),
                    coleg.get("nome"),
                )

                # 2) fallback: ente administrativo
                if not orgao_ult:
                    ente = best.get("enteAdministrativo") or {}
                    orgao_ult = self._concat_sigla_nome(
                        ente.get("sigla"),
                        ente.get("nome"),
                    )

                # 3) fallback final: ente de controle atual da autuação
                if not orgao_ult:
                    orgao_ult = self._concat_sigla_nome(
                        aut0.get("siglaEnteControleAtual"),
                        aut0.get("nomeEnteControleAtual"),
                    )

                return data_ult, orgao_ult, situacao_ult

        situacoes = aut0.get("situacoes") or []
        if isinstance(situacoes, list) and situacoes:
            best = None
            best_dt = None

            for it in situacoes:
                dt = self._parse_dt_flex(it.get("inicio"))
                if dt and (best_dt is None or dt > best_dt):
                    best_dt = dt
                    best = it

            if best and best_dt:
                data_ult = best_dt.strftime("%d/%m/%Y")
                situacao_ult = (best.get("descricao") or best.get("sigla") or "").strip()

                # 1) PRIORIDADE: colegiado
                coleg = best.get("colegiado") or {}
                orgao_ult = self._concat_sigla_nome(
                    coleg.get("sigla"),
                    coleg.get("nome"),
                )

                # 2) fallback: ente administrativo
                if not orgao_ult:
                    ente = best.get("enteAdministrativo") or {}
                    orgao_ult = self._concat_sigla_nome(
                        ente.get("sigla"),
                        ente.get("nome"),
                    )

                # 3) fallback final: ente de controle atual da autuação
                if not orgao_ult:
                    orgao_ult = self._concat_sigla_nome(
                        aut0.get("siglaEnteControleAtual"),
                        aut0.get("nomeEnteControleAtual"),
                    )

        return data_ult, orgao_ult, situacao_ult
   


    @staticmethod
    def _parse_dt_flex(value: Any) -> Optional[datetime]:
        if not value or pd.isna(value):
            return None
        s = str(value).strip().replace("T", " ")
        if "." in s:
            s = s.split(".")[0]
        for fmt in ("%Y-%m-%d %H:%M:%S", "%Y-%m-%d"):
            try:
                return datetime.strptime(s, fmt)
            except Exception:
                continue
        return None

    def fetch(self, sigla: str, numero: Any, ano: Any) -> Dict[str, Any]:
        pid = self.get_processo_id(sigla, numero, ano)
        if not pid:
            return {}

        if pid in self._detail_cache:
            return self._detail_cache[pid]

        try:
            js = self._get(f"processo/{pid}.json", params={"v": "1"})

            codigo_materia = self._deep_get_first(js, "codigoMateria")
            codigo_materia = str(codigo_materia).strip() if codigo_materia is not None else ""
            if codigo_materia.endswith(".0"):
                codigo_materia = codigo_materia[:-2]

            link_ficha = (
                f"https://www25.senado.leg.br/web/atividade/materias/-/materia/{codigo_materia}"
                if codigo_materia else ""
            )

            documento = js.get("documento") or {}
            autoria = documento.get("autoria") or []

            nomes = [
                a.get("autor") for a in autoria
                if isinstance(a, dict) and a.get("autor")
            ]
            propositor = ""
            if nomes:
                propositor = "; ".join(nomes[:5])
                if len(nomes) > 5:
                    propositor += f" (+{len(nomes)-5})"

            a0 = autoria[0] if isinstance(autoria, list) and autoria and isinstance(autoria[0], dict) else {}
            partido = (a0.get("siglaPartido") or "").strip()
            estado = (a0.get("uf") or "").strip()

            data_ult, orgao_ult, situacao_ult = self._extract_last_movement(js)

            url_documento = (documento.get("url") or "").strip()
            link_inteiro_teor = ""
            if url_documento:
                link_inteiro_teor = (
                    url_documento
                    if "disposition=" in url_documento
                    else f"{url_documento}&disposition=inline"
                )

            ementa = ""
            conteudo = js.get("conteudo") or {}
            if isinstance(conteudo, dict):
                ementa = (conteudo.get("ementa") or "").strip()
            if not ementa:
                ementa = str(self._deep_get_first(js, "ementa") or "").strip()

            out = {
                "senado_id_processo": pid,
                "senado_codigo_materia": codigo_materia,
                "senado_projeto": f"{str(sigla).strip().upper()} {int(numero)}/{int(ano)}",
                "senado_ementa": ementa,
                "senado_data_ultima_tramitacao": data_ult,
                "senado_orgao_ultima_tramitacao": orgao_ult,
                "senado_situacao_ultima_tramitacao": situacao_ult,
                "senado_data_parecer_aprovado": "",
                "senado_orgao_parecer": "",
                "senado_link_inteiro_teor_parecer": "",
                "senado_link_inteiro_teor_pl": link_inteiro_teor,
                "senado_link_ficha_tramitacao": link_ficha,
                "senado_data_proposta_pl": self._fmt_date(documento.get("dataApresentacao")),
                "senado_propositor_pl": propositor,
                "senado_partido": partido,
                "senado_estado": estado,
                "senado_emendas": link_ficha,
                "senado_substitutivos": link_ficha,
            }

            self._detail_cache[pid] = out
            return out

        except Exception:
            self._detail_cache[pid] = {}
            return {}

In [62]:
# Orquestrador Bicameral

class BicameralEnricher:
    def __init__(self, camara_client: CamaraClient, senado_client: SenadoClient):
        self.camara = camara_client
        self.senado = senado_client

    def enrich_reference(self, sigla: Any, numero: Any, ano: Any) -> Dict[str, Any]:
        camara_data = self.camara.fetch(sigla, numero, ano)
        time.sleep(0.05)  # suaviza requisições
        senado_data = self.senado.fetch(sigla, numero, ano)

        origem = "Não encontrado"
        if camara_data and senado_data:
            origem = "Câmara + Senado"
        elif camara_data:
            origem = "Câmara"
        elif senado_data:
            origem = "Senado"

        out = {
            "sigla": sigla,
            "numero": numero,
            "ano": ano,
            "Origem Dados": origem,
        }
        out.update(camara_data)
        out.update(senado_data)
        return out

    def enrich_dataframe(self, df: pd.DataFrame) -> pd.DataFrame:
        if not all(c in df.columns for c in ["sigla", "numero", "ano"]):
            raise KeyError("O dataframe precisa ter as colunas: sigla, numero, ano")

        referencias = (
            df[["sigla", "numero", "ano"]]
            .dropna(subset=["sigla", "numero", "ano"])
            .drop_duplicates()
            .copy()
        )

        resultados = []
        for _, row in referencias.iterrows():
            resultados.append(
                self.enrich_reference(row["sigla"], row["numero"], row["ano"])
            )

        return pd.DataFrame(resultados)

In [63]:
# Aplicação no Dataframe

# ============================================================
# 1. Extrai referência legislativa da coluna original
# ============================================================

parser_output = df_onedrive["Projeto de LEI"].apply(LegislativeReferenceParser.extract_reference)
parser_df = pd.DataFrame(list(parser_output))

df_onedrive["sigla"] = parser_df["sigla"]
df_onedrive["numero"] = parser_df["numero"]
df_onedrive["ano"] = parser_df["ano"]
df_onedrive["Projeto de Lei - Regex"] = parser_df["proposicao_normalizada"]

# tipos corretos
df_onedrive["numero"] = pd.to_numeric(df_onedrive["numero"], errors="coerce").astype("Int64")
df_onedrive["ano"] = pd.to_numeric(df_onedrive["ano"], errors="coerce").astype("Int64")


In [64]:
# ============================================================
# 2. Enriquecimento bicameral
# ============================================================

camara_client = CamaraClient()
senado_client = SenadoClient()
enricher = BicameralEnricher(camara_client, senado_client)

df_bicameral = enricher.enrich_dataframe(df_onedrive)

In [65]:
# ============================================================
# 3. Merge final
# ============================================================

df_final = pd.merge(
    df_onedrive,
    df_bicameral,
    on=["sigla", "numero", "ano"],
    how="left",
)

In [ ]:
# ============================================================
# 4. Normalização / organização
# ============================================================

df_final.columns = [c.strip() if isinstance(c, str) else c for c in df_final.columns]

if df_final.columns.duplicated().any():
    df_final = df_final.loc[:, ~df_final.columns.duplicated()]

colunas_principais = [
    "Projeto de LEI",
    "Projeto de Lei - Regex",
    "sigla",
    "numero",
    "ano",
    "Origem Dados",

    "camara_projeto",
    "camara_ementa",
    "camara_data_ultima_tramitacao",
    "camara_orgao_ultima_tramitacao",
    "camara_descricao_tramitacao",
    "camara_situacao_ultima_tramitacao",
    "camara_despacho_ultima_tramitacao",
    "camara_data_parecer_aprovado",
    "camara_orgao_parecer",
    "camara_despacho_parecer",
    "camara_link_inteiro_teor_parecer",
    "camara_link_inteiro_teor_pl",
    "camara_link_ficha_tramitacao",
    "camara_data_proposta_pl",
    "camara_propositor_pl",
    "camara_partido",
    "camara_estado",
    "camara_emendas",
    "camara_substitutivos",

    "senado_projeto",
    "senado_ementa",
    "senado_data_ultima_tramitacao",
    "senado_orgao_ultima_tramitacao",
    "senado_situacao_ultima_tramitacao",
    "senado_data_parecer_aprovado",
    "senado_orgao_parecer",
    "senado_link_inteiro_teor_parecer",
    "senado_link_inteiro_teor_pl",
    "senado_link_ficha_tramitacao",
    "senado_data_proposta_pl",
    "senado_propositor_pl",
    "senado_partido",
    "senado_estado",
    "senado_emendas",
    "senado_substitutivos",
]

colunas_principais_ok = [c for c in colunas_principais if c in df_final.columns]
df_final = df_final[colunas_principais_ok + [c for c in df_final.columns if c not in colunas_principais_ok]]

In [67]:
# =========================================================
# 4. Normalização / Organização
# =========================================================

# ---------------------------
# Padronização de datas
# ---------------------------
colunas_data = [
    "camara_data_ultima_tramitacao",
    "camara_data_parecer_aprovado",
    "camara_data_proposta_pl",
    "senado_data_ultima_tramitacao",
    "senado_data_parecer_aprovado",
    "senado_data_proposta_pl",
    "1º Encaminhamento - Data",
    "1º Encaminhamento - Prazo p/ Resposta",
    "2º Encaminhamento - Data",
    "2º Encaminhamento - Prazo p/ Resposta",
    "3º Encaminhamento - Data",
    "3º Encaminhamento - Prazo p/ Resposta",
    "Envio para a Delog - Data",
    "Envio ao Gabinete da Seges - Data",
    "Assinaturas - Data",
]


# Aqui a data da ultima tramitação no Senado estava trazendo a data invertida.
# for col in colunas_data:
#     if col in df_final.columns:
#         df_final[col] = (
#             pd.to_datetime(df_final[col], errors="coerce")
#             .dt.strftime("%d/%m/%Y")
#             .fillna("")
#         )

#Trecho corrigido
for col in colunas_data:
    if col in df_final.columns:
        df_final[col] = (
            pd.to_datetime(df_final[col], errors="coerce", dayfirst=True)
            .dt.strftime("%d/%m/%Y")
            .fillna("")
        )


# ---------------------------
# Ajuste de colunas numéricas inteiras para não exibir .0
# ---------------------------
colunas_inteiras = [
    "numero",
    "ano",
    "Validação pelo gabinete da Seges - Quantidade de dias úteis",
    "camara_id_proposicao",
    "senado_id_processo",
    "senado_codigo_materia",
]

for col in colunas_inteiras:
    if col in df_final.columns:
        df_final[col] = pd.to_numeric(df_final[col], errors="coerce").astype("Int64")



C:\Users\Rafael\AppData\Local\Temp\ipykernel_29080\1587745853.py:40: UserWarning: Parsing dates in %Y-%m-%dT%H:%M format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(df_final[col], errors="coerce", dayfirst=True)
C:\Users\Rafael\AppData\Local\Temp\ipykernel_29080\1587745853.py:40: UserWarning: Parsing dates in %Y-%m-%dT%H:%M format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(df_final[col], errors="coerce", dayfirst=True)


In [68]:
# ============================================================
# 5. Exportação
# ============================================================

df_final.to_csv("data/df_final_bicameral.csv", index=False, encoding="utf-8-sig")
